Preparations:
- load prediction.csv
    - drop the lines with the non-used ifs
    - if there is more then one if, take mean

- add column "sample" to master table
- join them
- drop unnecessary columns (matadata, paths, ranks)

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import ast
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, f1_score, precision_recall_curve, confusion_matrix

# Configuration
CSV_PATH =  "/home/bri/pymol/data/predictions/preds_harvey_org.csv" 
MASTER_PATH =  "/home/bri/pymol/data/harvey/harvey.csv" 
YAML_PATH = "/home/bri/pymol/EDA/static/metric_directions.yaml"
OUTPUT_DIR = Path("/home/bri/pymol/EDA/static/harvey_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


KEEP_INTERFACES = {"A,C", "B,C"}   # as option if there is more than one, default A,B



# Load data
df_if = pd.read_csv(CSV_PATH)
df_if.info()


<class 'pandas.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 92 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   sample                                  37 non-null     str    
 1   binding                                 37 non-null     bool   
 2   interface                               37 non-null     str    
 3   af3_pae                                 37 non-null     float64
 4   af3_plddt                               37 non-null     float64
 5   af3_interface_dG                        37 non-null     float64
 6   af3_interface_dG_SASA_ratio             37 non-null     float64
 7   af3_ipae                                37 non-null     float64
 8   af3_iplddt                              37 non-null     float64
 9   af3_ipsae                               37 non-null     float64
 10  af3_pdockq                              37 non-null     float64
 11  af3_pd

#### all

In [11]:
num_cols = df_if.select_dtypes(include=['number']).columns
other_cols = [c for c in df_if.columns if c not in num_cols and c != 'sample']

agg_dict = {col: 'mean' for col in num_cols}
agg_dict.update({col: 'first' for col in other_cols})

df_1if = df_if.groupby('sample').agg(agg_dict).reset_index()

df_1if

,sample,af3_pae,af3_plddt,af3_interface_dG,af3_interface_dG_SASA_ratio,af3_ipae,af3_iplddt,af3_ipsae,af3_pdockq,af3_pdockq2,...,binding,interface,chai_unconstrained_model_path,chai_constrained_model_path,chai_unconstrained_per_chain_ptm,chai_unconstrained_per_chain_pair_iptm,chai_constrained_per_chain_ptm,chai_constrained_per_chain_pair_iptm,boltz_free_model_path,boltz_template_model_path
0,AT118i4h32_AGTR1,11.528240,78.049861,241.77,7.91,7.356923,76.922909,0.640211,0.4392,0.2198,...,False,"A,B",/scicore/home/schwede/barta0000/harvey_ds/orig...,/scicore/home/schwede/barta0000/harvey_ds/orig...,"[[0.8844592571258545, 0.7967235445976257]]","[[[0.8844592571258545, 0.45740675926208496], [...","[[0.899858832359314, 0.9029654264450073]]","[[[0.899858832359314, 0.7897694110870361], [0....",/scicore/home/schwede/barta0000/harvey_ds/orig...,/scicore/home/schwede/barta0000/harvey_ds/orig...
1,AT201_AGTR1,12.212700,76.021409,108.25,7.19,18.501786,78.594062,0.426000,0.1468,0.1714,...,False,"A,B",/scicore/home/schwede/barta0000/harvey_ds/orig...,/scicore/home/schwede/barta0000/harvey_ds/orig...,"[[0.885317325592041, 0.7817205786705017]]","[[[0.885317325592041, 0.19592402875423431], [0...","[[0.8954206705093384, 0.82906174659729]]","[[[0.8954206705093384, 0.5672476887702942], [0...",/scicore/home/schwede/barta0000/harvey_ds/orig...,/scicore/home/schwede/barta0000/harvey_ds/orig...
2,AT202_AGTR1,13.119915,76.458147,299.89,11.87,17.030000,68.662117,0.219051,0.1821,0.0583,...,False,"A,B",/scicore/home/schwede/barta0000/harvey_ds/orig...,/scicore/home/schwede/barta0000/harvey_ds/orig...,"[[0.8855280876159668, 0.7832863926887512]]","[[[0.8855280876159668, 0.22815226018428802], [...","[[0.8752248883247375, 0.8392460346221924]]","[[[0.8752248883247375, 0.5595718622207642], [0...",/scicore/home/schwede/barta0000/harvey_ds/orig...,/scicore/home/schwede/barta0000/harvey_ds/orig...
3,AT203_AGTR1,11.603302,78.053494,77.77,2.87,8.088182,80.329839,0.647259,0.4361,0.3651,...,False,"A,B",/scicore/home/schwede/barta0000/harvey_ds/orig...,/scicore/home/schwede/barta0000/harvey_ds/orig...,"[[0.862318754196167, 0.8046218156814575]]","[[[0.862318754196167, 0.37608620524406433], [0...","[[0.8754035830497742, 0.8123531937599182]]","[[[0.8754035830497742, 0.6163707971572876], [0...",/scicore/home/schwede/barta0000/harvey_ds/orig...,/scicore/home/schwede/barta0000/harvey_ds/orig...
4,AT204_AGTR1,16.662872,71.692176,456.19,24.69,14.596774,63.245793,0.000000,0.0919,0.0115,...,False,"A,B",/scicore/home/schwede/barta0000/harvey_ds/orig...,/scicore/home/schwede/barta0000/harvey_ds/orig...,"[[0.9009032845497131, 0.7990964651107788]]","[[[0.9009032845497131, 0.2246391326189041], [0...","[[0.9041344523429871, 0.8222752809524536]]","[[[0.9041344523429871, 0.6109544038772583], [0...",/scicore/home/schwede/barta0000/harvey_ds/orig...,/scicore/home/schwede/barta0000/harvey_ds/orig...
5,AT205_AGTR1,12.078681,77.945758,333.55,11.66,16.094737,72.127838,0.305816,0.2805,0.1133,...,False,"A,B",/scicore/home/schwede/barta0000/harvey_ds/orig...,/scicore/home/schwede/barta0000/harvey_ds/orig...,"[[0.8954852819442749, 0.7971125841140747]]","[[[0.8954852819442749, 0.31081849336624146], [...","[[0.9003657102584839, 0.8154690265655518]]","[[[0.9003657102584839, 0.6001021265983582], [0...",/scicore/home/schwede/barta0000/harvey_ds/orig...,/scicore/home/schwede/barta0000/harvey_ds/orig...
6,AT206_AGTR1,12.140000,77.104133,565.08,17.43,7.664615,73.026588,0.429742,0.3425,0.1524,...,False,"A,B",/scicore/home/schwede/barta0000/harvey_ds/orig...,/scicore/home/schwede/barta0000/harvey_ds/orig...,"[[0.8934987187385559, 0.8168030977249146]]","[[[0.8934987187385559, 0.6189501285552979], [0...","[[0.9000794887542725, 0.8263767957687378]]","[[[0.9000794887542725, 0.6687699556350708], [0...",/scicore/home/schwede/barta0000/harvey_ds/orig...,/scicore/home/schwede/barta0000/harvey_ds/orig...
7,AT207_AGTR1,14.515562,72.984429,575.85,17.79,13.043966,64.423951,0.127196,0.2251,0.0204,...,False,"A,B",/scicore/

In [12]:
df_2if = df_1if.drop(columns=["binding", "interface","chai_unconstrained_model_path", "chai_constrained_model_path", "boltz_free_model_path", "boltz_template_model_path"])
df_2if

,sample,af3_pae,af3_plddt,af3_interface_dG,af3_interface_dG_SASA_ratio,af3_ipae,af3_iplddt,af3_ipsae,af3_pdockq,af3_pdockq2,...,boltz_template_ipde,boltz_template_per_chain_pair_iptm,boltz_template_ipsae,boltz_template_pdockq,boltz_template_pdockq2,boltz_template_lis,chai_unconstrained_per_chain_ptm,chai_unconstrained_per_chain_pair_iptm,chai_constrained_per_chain_ptm,chai_constrained_per_chain_pair_iptm
0,AT118i4h32_AGTR1,11.528240,78.049861,241.77,7.91,7.356923,76.922909,0.640211,0.4392,0.2198,...,0.649316,0.953019,0.871298,0.5023,0.5615,0.6842,"[[0.8844592571258545, 0.7967235445976257]]","[[[0.8844592571258545, 0.45740675926208496], [...","[[0.899858832359314, 0.9029654264450073]]","[[[0.899858832359314, 0.7897694110870361], [0...."
1,AT201_AGTR1,12.212700,76.021409,108.25,7.19,18.501786,78.594062,0.426000,0.1468,0.1714,...,2.173097,0.910365,0.755096,0.4618,0.3734,0.5413,"[[0.885317325592041, 0.7817205786705017]]","[[[0.885317325592041, 0.19592402875423431], [0...","[[0.8954206705093384, 0.82906174659729]]","[[[0.8954206705093384, 0.5672476887702942], [0..."
2,AT202_AGTR1,13.119915,76.458147,299.89,11.87,17.030000,68.662117,0.219051,0.1821,0.0583,...,4.002924,0.822987,0.587241,0.6138,0.2044,0.4111,"[[0.8855280876159668, 0.7832863926887512]]","[[[0.8855280876159668, 0.22815226018428802], [...","[[0.8752248883247375, 0.8392460346221924]]","[[[0.8752248883247375, 0.5595718622207642], [0..."
3,AT203_AGTR1,11.603302,78.053494,77.77,2.87,8.088182,80.329839,0.647259,0.4361,0.3651,...,1.497975,0.910428,0.795326,0.6598,0.6871,0.5950,"[[0.862318754196167, 0.8046218156814575]]","[[[0.862318754196167, 0.37608620524406433], [0...","[[0.8754035830497742, 0.8123531937599182]]","[[[0.8754035830497742, 0.6163707971572876], [0..."
4,AT204_AGTR1,16.662872,71.692176,456.19,24.69,14.596774,63.245793,0.000000,0.0919,0.0115,...,3.520961,0.815329,0.557721,0.5336,0.2333,0.4120,"[[0.9009032845497131, 0.7990964651107788]]","[[[0.9009032845497131, 0.2246391326189041], [0...","[[0.9041344523429871, 0.8222752809524536]]","[[[0.9041344523429871, 0.6109544038772583], [0..."
5,AT205_AGTR1,12.078681,77.945758,333.55,11.66,16.094737,72.127838,0.305816,0.2805,0.1133,...,3.379495,0.821988,0.592587,0.5518,0.4039,0.4504,"[[0.8954852819442749, 0.7971125841140747]]","[[[0.8954852819442749, 0.31081849336624146], [...","[[0.9003657102584839, 0.8154690265655518]]","[[[0.9003657102584839, 0.6001021265983582], [0..."
6,AT206_AGTR1,12.140000,77.104133,565.08,17.43,7.664615,73.026588,0.429742,0.3425,0.1524,...,1.158381,0.935807,0.820640,0.6315,0.7949,0.6180,"[[0.8934987187385559, 0.8168030977249146]]","[[[0.8934987187385559, 0.6189501285552979], [0...","[[0.9000794887542725, 0.8263767957687378]]","[[[0.9000794887542725, 0.6687699556350708], [0..."
7,AT207_AGTR1,14.515562,72.984429,575.85,17.79,13.043966,64.423951,0.127196,0.2251,0.0204,...,1.897490,0.902147,0.745535,0.6241,0.5803,0.5907,"[[0.9021723866462708, 0.7938095927238464]]","[[[0.9021723866462708, 0.48009294271469116], [...","[[0.9140447974205017, 0.8120380640029907]]","[[[0.9140447974205017, 0.5632482767105103], [0..."
8,AT208_AGTR1,13.400510,74.515406,489.56,14.97,12.274658,68.717253,0.228331,0.2595,0.0610,...,2.079581,0.835657,0.688378,0.6490,0.4953,0.4684,"[[0.895566463470459, 0.8259811401367188]]","[[[0.895566463470459, 0.6455579996109009], [0....","[[0.8955528140068054, 0.8270229697227478]]","[[[0.8955528140068054, 0.6786258816719055], [0..."
9,AT209_AGTR1,14.760581,72.510105,671.46,20.51,14.528947,62.970591,0.121292,0.2023,0.0264,...,1.618617,0.901725,0.766039,0.5527,0.5443,0.5774,"[[0.8895854353904724, 0.8035908341407776]]","[[[0.8895854353904724, 0.43610429763793945], [...","[[0.9006621241569519, 0.8193023204803467]]","[[[0.9006621241569519, 0.7213302850723267], [0..."


## master df

##### snir


In [5]:
# for snir data
master_df = pd.read_csv(MASTER_PATH, delimiter=';')
master_df

,source,type,sample,fab,interface,vh/vl,target,target_residues,target_sequence,binder,KD[M],notes
0,snir_ab,original,5ifj_abc_cut,1000.0,"A,C",EVQLLESGGGLVQPGGSLRLSCAASGFTFSTYAMSWVRQAPGKGLE...,gliadin_alpha_E,NaN,PLQPEQPFP,1,0.000010,Fab 1E03 + KD[M] approximated from Snir et. al...
1,snir_ab,original,5ifj_abc_cut,1000.0,"B,C",QLVLTQSPSASASLGASVRLTCTLSSGHSSYVIAWHQQQSEKGPRY...,gliadin_alpha_E,NaN,PLQPEQPFP,1,0.000010,Fab 1E03 + KD[M] approximated from Snir et. al...
2,snir_ab,original,5ig7_abc_cut,10.0,"A,C",EVQLLESGGGLVQPGGSLRLSCAASGFTFSTYAMSWVRQAPGKGLE...,gliadin_alpha_Q,NaN,PLQPQQPFP,1,0.000100,Fab 1E01 + KD[M] approximated from Snir et. al...
3,snir_ab,original,5ig7_abc_cut,10.0,"B,C",QLVLTQSPSASASLGASVRLTCTLSSGHSSYVIAWHQQQSEKGPRY...,gliadin_alpha_Q,NaN,PLQPQQPFP,1,0.000100,Fab 1E01 + KD[M] approximated from Snir et. al...
4,snir_ab,original,5ijk_acx_cut,10.0,"A,C",EVQLVESGGGLVKPGGSLRLSCAASGFTFSNAWFNWVRQAPGKGLE...,gliadin_gamma_E,NaN,LQPEQPFP,1,0.000001,Fab 1E01 + KD[M] approximated from Snir et. al...
...,...,...,...,...,...,...,...,...,...,...,...,...
65,snir_ab,alanine_scan,5ijk_acx_cut_B_Y108A,10.0,"B,C",DIVMTQSPDSLAVSLGERATINCKSSQSVLYSSNNKNYLAWYQQKP...,gliadin_gamma_E,NaN,LQPEQPFP,0,NaN,NaN
66,snir_ab,alanine_scan,5ijk_acx_cut_B_Y31A,10.0,"A,C",EVQLVESGGGLVKPGGSLRLSCAASGFTFSNAWFNWVRQAPGKGLE...,gliadin_gamma_E,NaN,LQPEQPFP,0,NaN,NaN
67,snir_ab,alanine_scan,5ijk_acx_cut_B_Y31A,10.0,"B,C",DIVMTQSPDSLAVSLGERATINCKSSQSVLASSNNKNYLAWYQQKP...,gliadin_gamma_E,NaN,LQPEQPFP,0,NaN,NaN
68,snir_ab,alanine_scan,5ijk_acx_cut_B_Y38A,10.0,"A,C",EVQLVESGGGLVKPGGSLRLSCAASGFTFSNAWFNWVRQAPGKGLE...,gliadin_gamma_E,NaN,LQPEQPFP,0,NaN,NaN


In [6]:
master_df2 = master_df.drop(columns=["interface", "fab", "vh/vl", "target", "target_residues", "target_sequence", "notes"])
master_df2

,source,type,sample,binder,KD[M]
0,snir_ab,original,5ifj_abc_cut,1,0.000010
1,snir_ab,original,5ifj_abc_cut,1,0.000010
2,snir_ab,original,5ig7_abc_cut,1,0.000100
3,snir_ab,original,5ig7_abc_cut,1,0.000100
4,snir_ab,original,5ijk_acx_cut,1,0.000001
...,...,...,...,...,...
65,snir_ab,alanine_scan,5ijk_acx_cut_B_Y108A,0,NaN
66,snir_ab,alanine_scan,5ijk_acx_cut_B_Y31A,0,NaN
67,snir_ab,alanine_scan,5ijk_acx_cut_B_Y31A,0,NaN
68,snir_ab,alanine_scan,5ijk_acx_cut_B_Y38A,0,NaN


In [7]:
master_df3 = master_df2.drop_duplicates(keep="last")
master_df3

,source,type,sample,binder,KD[M]
1,snir_ab,original,5ifj_abc_cut,1,0.000010
3,snir_ab,original,5ig7_abc_cut,1,0.000100
5,snir_ab,original,5ijk_acx_cut,1,0.000001
7,snir_ab,alanine_scan,5ifj_abc_cut_A_D111A,0,NaN
9,snir_ab,alanine_scan,5ifj_abc_cut_A_K110A,0,NaN
11,snir_ab,alanine_scan,5ifj_abc_cut_A_S64A,0,NaN
13,snir_ab,alanine_scan,5ifj_abc_cut_A_W52A,0,NaN
15,snir_ab,alanine_scan,5ifj_abc_cut_A_Y66A,0,NaN
17,snir_ab,alanine_scan,5ifj_abc_cut_all_A,0,NaN
19,snir_ab,alanine_scan,5ifj_abc_cut_B_L116A,0,NaN


##### mcmahon / germinal

In [13]:

master_df = pd.read_csv(MASTER_PATH, delimiter=';')
master_df.head()

,source,type,vhh,vhh_sequence,target,target_residues,target_sequence,binder,KD[M],EC50[M],notes
0,harvey,original,AT118i4h32,EVQLVESGGGLVQPGGSLRLSCAASGYIYRRYRMGWYRQAPGKGRE...,AGTR1,1-359,MILNSSTEDGIKRIQDDCPKAGRHNYIFVMIPTLYSIIFVVGIFGN...,1.0,NaN,NaN,8TH3
1,harvey,original,AT201,QVQLQESGGGLVQAGGSLRLSCAASGTISDPPYMGWYRQAPGKERE...,AGTR1,1-359,MILNSSTEDGIKRIQDDCPKAGRHNYIFVMIPTLYSIIFVVGIFGN...,1.0,NaN,NaN,NaN
2,harvey,original,AT202,QVQLQESGGGLVQAGGSLRLSCAASGYIFDLPYMGWYRQAPGKERE...,AGTR1,1-359,MILNSSTEDGIKRIQDDCPKAGRHNYIFVMIPTLYSIIFVVGIFGN...,1.0,NaN,NaN,NaN
3,harvey,original,AT203,QVQLQESGGLVQAGGSLRLSCAASGYISRSYDMGWYRQAPGKEREL...,AGTR1,1-359,MILNSSTEDGIKRIQDDCPKAGRHNYIFVMIPTLYSIIFVVGIFGN...,1.0,NaN,NaN,NaN
4,harvey,original,AT204,QVQLQESGGGLVQAGGSLRLSCAASGYIFTRGFMGWYRQAPGKERE...,AGTR1,1-359,MILNSSTEDGIKRIQDDCPKAGRHNYIFVMIPTLYSIIFVVGIFGN...,1.0,NaN,NaN,NaN


In [14]:
AMBIGUOUS_TARGET= "HSA"   # test

def create_sample_id(row):
    vhh = str(row['vhh'])
    target = str(row['target'])
    residues = str(row['target_residues'])
    
    # Check if target is (ambigous target, meaning the same target has different version --> target residues are needed to be able to identify it, the only one for now is HSA) to apply the specific naming convention
    if target == AMBIGUOUS_TARGET:
        return f"{vhh}_{target}_{residues}"
    else:
        return f"{vhh}_{target}"

# 1. Create the 'sample' column in the master table
master_df['sample'] = master_df.apply(create_sample_id, axis=1)
master_df



,source,type,vhh,vhh_sequence,target,target_residues,target_sequence,binder,KD[M],EC50[M],notes,sample
0,harvey,original,AT118i4h32,EVQLVESGGGLVQPGGSLRLSCAASGYIYRRYRMGWYRQAPGKGRE...,AGTR1,1-359,MILNSSTEDGIKRIQDDCPKAGRHNYIFVMIPTLYSIIFVVGIFGN...,1.0,NaN,NaN,8TH3,AT118i4h32_AGTR1
1,harvey,original,AT201,QVQLQESGGGLVQAGGSLRLSCAASGTISDPPYMGWYRQAPGKERE...,AGTR1,1-359,MILNSSTEDGIKRIQDDCPKAGRHNYIFVMIPTLYSIIFVVGIFGN...,1.0,NaN,NaN,NaN,AT201_AGTR1
2,harvey,original,AT202,QVQLQESGGGLVQAGGSLRLSCAASGYIFDLPYMGWYRQAPGKERE...,AGTR1,1-359,MILNSSTEDGIKRIQDDCPKAGRHNYIFVMIPTLYSIIFVVGIFGN...,1.0,NaN,NaN,NaN,AT202_AGTR1
3,harvey,original,AT203,QVQLQESGGLVQAGGSLRLSCAASGYISRSYDMGWYRQAPGKEREL...,AGTR1,1-359,MILNSSTEDGIKRIQDDCPKAGRHNYIFVMIPTLYSIIFVVGIFGN...,1.0,NaN,NaN,NaN,AT203_AGTR1
4,harvey,original,AT204,QVQLQESGGGLVQAGGSLRLSCAASGYIFTRGFMGWYRQAPGKERE...,AGTR1,1-359,MILNSSTEDGIKRIQDDCPKAGRHNYIFVMIPTLYSIIFVVGIFGN...,1.0,NaN,NaN,NaN,AT204_AGTR1
5,harvey,original,AT205,QVQLQESGGGLVQAGGSLRLSCAASGYIFYDSVMGWYRQAPGKERE...,AGTR1,1-359,MILNSSTEDGIKRIQDDCPKAGRHNYIFVMIPTLYSIIFVVGIFGN...,1.0,NaN,NaN,NaN,AT205_AGTR1
6,harvey,original,AT206,QVQLQESGGGLVQAGGSLRLSCAASGSISYYRMGWYRQAPGKERKF...,AGTR1,1-359,MILNSSTEDGIKRIQDDCPKAGRHNYIFVMIPTLYSIIFVVGIFGN...,1.0,NaN,NaN,NaN,AT206_AGTR1
7,harvey,original,AT207,QVQLQESGGGLVQAGGSLRLSCAASGYIFTHSSMGWYRQAPGKERE...,AGTR1,1-359,MILNSSTEDGIKRIQDDCPKAGRHNYIFVMIPTLYSIIFVVGIFGN...,1.0,NaN,NaN,NaN,AT207_AGTR1
8,harvey,original,AT208,QVQLQESGGGLVQAGGSLRLSCAASGYIFTRYRMGWYRQAPGKERE...,AGTR1,1-359,MILNSSTEDGIKRIQDDCPKAGRHNYIFVMIPTLYSIIFVVGIFGN...,1.0,NaN,NaN,NaN,AT208_AGTR1
9,harvey,original,AT209,QVQLQESGGGLVQAGGSLRLSCAASGYIFYRYSMGWYRQAPGKERE...,AGTR1,1-359,MILNSSTEDGIKRIQDDCPKAGRHNYIFVMIPTLYSIIFVVGIFGN...,1.0,NaN,NaN,NaN,AT209_AGTR1


In [15]:
master_df2 = master_df.drop(columns=["vhh", "vhh_sequence", "target", "target_residues", "target_sequence", "notes"])
master_df2

,source,type,binder,KD[M],EC50[M],sample
0,harvey,original,1.0,NaN,NaN,AT118i4h32_AGTR1
1,harvey,original,1.0,NaN,NaN,AT201_AGTR1
2,harvey,original,1.0,NaN,NaN,AT202_AGTR1
3,harvey,original,1.0,NaN,NaN,AT203_AGTR1
4,harvey,original,1.0,NaN,NaN,AT204_AGTR1
5,harvey,original,1.0,NaN,NaN,AT205_AGTR1
6,harvey,original,1.0,NaN,NaN,AT206_AGTR1
7,harvey,original,1.0,NaN,NaN,AT207_AGTR1
8,harvey,original,1.0,NaN,NaN,AT208_AGTR1
9,harvey,original,1.0,NaN,NaN,AT209_AGTR1


### merge

In [16]:
scores_df = pd.merge(master_df2, df_2if, on=['sample'], how='inner')
#df = pd.merge(master_df2, preds_df, on=['sample'], how='inner')
# Display the first few rows to verify
print(scores_df[['sample', 'af3_plddt']])

# Save the merged result
scores_df.to_csv(OUTPUT_DIR /'merged_harvey.csv', index=False)

                    sample  af3_plddt
0         AT118i4h32_AGTR1  78.049861
1              AT201_AGTR1  76.021409
2              AT202_AGTR1  76.458147
3              AT203_AGTR1  78.053494
4              AT204_AGTR1  71.692176
5              AT205_AGTR1  77.945758
6              AT206_AGTR1  77.104133
7              AT207_AGTR1  72.984429
8              AT208_AGTR1  74.515406
9              AT209_AGTR1  72.510105
10             AT210_AGTR1  76.180237
11             AT211_AGTR1  72.819037
12             AT212_AGTR1  74.538897
13             AT213_AGTR1  75.139875
14                CH1_ACM3  78.071409
15                CH2_ACM3  78.288319
16              Nb29_AA2AR  79.137338
17        Sim0563_MRGPCRX2  79.174400
18        Sim1846_MRGPCRX2  77.182731
19        Sim3014_MRGPCRX2  75.173242
20        Sim4177_MRGPCRX2  74.847496
21        Sim4717_MRGPCRX2  78.896985
22        Sim4784_MRGPCRX2  76.737652
23        Sim7252_MRGPCRX2  76.717782
24        Sim7492_MRGPCRX2  74.520081
25        Si